# Kinoloom — smoke test (T4 feasibility check)

Answers, on a real free-tier T4, the questions that decide the whole Colab rollout:
1. Does the **H3 W4A8 int8 (ConvRot)** stack load and render on **sm_75** (Turing, no bf16)? I have only run it on an RTX 3070 (sm_86).
2. What is the **VRAM peak** and **seconds/frame** at 960×544?
3. Do the pinned ComfyUI v0.36 tree + custom nodes import cleanly on Linux?
4. How fast is the **HF pull** (decides per-session boot time)?

Secrets needed: `HF_TOKEN`. `TAILSCALE_AUTHKEY` is optional and enables the mesh probe cell; `GITHUB_PAT` is needed only to clone a private fork.

**Prerequisite:** the W4A8 weights must be published first — on the PC: `python tool/scripts/publish_models_hf.py --groups h3_shared,h3_dit_singularity --repo datasets/<you>/kinoloom-models`.

Verdict rule: if the render cell completes with ≤ 16 GB peak → keep the bootstrap default `MODEL_GROUPS = h3_shared,h3_dit_singularity`. If the kernels fail → H3 (and the W4A8 `ltx_core` stack) stay on the 3070; `install/colab_model_groups.json` has no Wan group, so there is no video fallback group for Colab.

In [ ]:
# @title 1 · Environment report { display-mode: "form" }
import os, torch, shutil
print("torch          :", torch.__version__, "| cuda", torch.version.cuda)
print("device         :", torch.cuda.get_device_name(0))
print("capability     : sm_%d%d" % torch.cuda.get_device_capability(0))
print("bf16 supported :", torch.cuda.is_bf16_supported())
print("VRAM           : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
u = shutil.disk_usage("/content")
print("disk           : %.1f GB free of %.1f GB" % (u.free / 2**30, u.total / 2**30))

def secret(name):
    try:
        from google.colab import userdata
        try: return userdata.get(name) or ""
        except Exception: return ""
    except Exception:
        return os.environ.get(name, "")

for k in ("GITHUB_PAT", "HF_TOKEN", "TAILSCALE_AUTHKEY"):
    os.environ[k] = secret(k) or os.environ.get(k, "")
missing = [k for k in ("HF_TOKEN",) if not os.environ.get(k)]
if missing:
    raise SystemExit(f"missing secrets: {missing}")
print("secrets OK")

In [ ]:
# @title 2 · Clone repo + build ComfyUI stack (SMOKE mode — no worker agent) { display-mode: "form" }
import os, subprocess
AS_ROOT = "/content/kinoloom"
REPO = "github.com/harlixay7/Kinoloom.git"
if not os.path.isdir(f"{AS_ROOT}/.git"):
    pat = os.environ.get("GITHUB_PAT", "")   # only for a private fork
    url = f"https://x-access-token:{pat}@{REPO}" if pat else f"https://{REPO}"
    subprocess.run(["git", "clone", url, AS_ROOT], check=True)
    if pat:   # never keep the PAT in .git/config
        subprocess.run(["git", "-C", AS_ROOT, "remote", "set-url", "origin",
                        f"https://{REPO}"], check=True)
os.environ["SMOKE"] = "1"
os.environ.setdefault("MODEL_GROUPS", "h3_shared,h3_dit_singularity")
!bash /content/kinoloom/colab/bootstrap.sh

In [ ]:
# @title 3 · Start ComfyUI, verify H3 node classes { display-mode: "form" }
import subprocess, time, json, urllib.request
subprocess.run(["pkill", "-f", "ComfyUI/main.py"], capture_output=True)
log = open("/content/kinoloom/colab/comfyui.log", "ab")
subprocess.Popen(["/content/kinoloom/.colab-venv/bin/python", "main.py",
                  "--port", "8188", "--default-device", "0",
                  "--disable-auto-launch"],
                 cwd="/content/kinoloom/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
def get(path, timeout=10):
    with urllib.request.urlopen(f"http://127.0.0.1:8188{path}", timeout=timeout) as r:
        return json.loads(r.read().decode())
for i in range(60):
    try:
        get("/system_stats"); break
    except Exception:
        time.sleep(5)
else:
    print("COMFY DID NOT START — first 60 log lines:")
    print(open("/content/kinoloom/colab/comfyui.log", errors="replace").read()[-4000:])
    raise SystemExit(1)
info = get("/object_info", timeout=180)
need = ["UNETLoader", "CLIPLoader", "VAELoader",
        "MiniMaxH3ReferenceToVideo", "ModelAttentionBackend",
        "MiniMaxH3SigmaShift", "ManualSigmas", "SamplerCustomAdvanced",
        "CreateVideo", "SaveVideo"]
have = {n: n in info for n in need}
for n, ok in have.items():
    print(f"  {n:32s} {'OK' if ok else 'MISSING'}")
if not all(have.values()):
    print("\nmissing node classes — check custom-node import errors:")
    print(open("/content/kinoloom/colab/comfyui.log", errors="replace").read()[-4000:])
    raise SystemExit(1)
print("ComfyUI up, all H3 node classes present")

In [ ]:
# @title 4 · THE GATE: one W4A8 render at 960×544 on the T4 { display-mode: "form" }
import json, re, subprocess, threading, time, urllib.request

tpl = open("/content/kinoloom/tool/workflows/h3_t2v.json", encoding="utf-8").read()
subs = {"MODEL_5B": "Minimax-h3_Singularity_ref2va_v1.3_Pruned_w4a8.safetensors",
        "H3_TE": "qwen3vl_32b_minimax_h3-w4a8_convrot.safetensors",
        "H3_VAE_VIDEO": "minimax_h3_video_vae_fp16.safetensors",
        "H3_VAE_AUDIO": "minimax_h3_audio_vae_fp32.safetensors",
        "WIDTH": "960", "HEIGHT": "544", "STEPS": "8", "FRAMES": "22",
        "PROMPT": "a slow dolly shot across a neon-lit city street at night, cinematic",
        "SEED": "42", "SHOT_ID": "smoke_t4"}
for k, v in subs.items():
    tpl = tpl.replace("{%s}" % k, json.dumps(v)[1:-1])   # JSON-escape like workflows.py
left = sorted(set(re.findall(r"\{[A-Z_0-9]+\}", tpl)))
assert not left, f"unsubstituted tokens: {left}"
workflow = json.loads(tpl)

def post(path, data):
    req = urllib.request.Request(f"http://127.0.0.1:8188{path}",
          data=json.dumps(data).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read().decode())

peak = {"mb": 0}
stop = threading.Event()
def watch():
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip()
        try: peak["mb"] = max(peak["mb"], int(out.splitlines()[0]))
        except Exception: pass
        time.sleep(2)
threading.Thread(target=watch, daemon=True).start()

t0 = time.time()
pid = post("/prompt", {"prompt": workflow, "client_id": "smoke"})["prompt_id"]
print("prompt:", pid, "— rendering 960x544, 8 steps, 22 frames …")
deadline = time.time() + 45 * 60
while time.time() < deadline:
    h = get(f"/history/{pid}", timeout=30)
    if pid in h:
        break
    time.sleep(10)
else:
    raise SystemExit("TIMEOUT — see /content/kinoloom/colab/comfyui.log")
secs = time.time() - t0
stop.set(); time.sleep(3)
entry = h[pid]
st = entry.get("status", {})
errs = [m[1].get("exception_message", "")[:200]
        for m in st.get("messages", []) if m[0] == "execution_error"]
outs = [o for node in (entry.get("outputs") or {}).values()
        for key in ("images", "gifs", "videos") for o in node.get(key, [])]
result = {"gpu": "T4 sm_75", "verdict": "PASS" if (st.get("status_str") == "success" and not errs) else "FAIL",
          "seconds": round(secs, 1), "frames": 22, "steps": 8,
          "vram_peak_gb": round(peak["mb"] / 1024, 2),
          "s_per_frame": round(secs / 22, 2),
          "errors": errs, "outputs": outs[:3]}
open("/content/kinoloom/colab/smoke_result.json", "w").write(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
print("\nVERDICT:", result["verdict"],
      "— copy colab/smoke_result.json into the repo and record the T4 benchmark entry"
      if result["verdict"] == "PASS" else
      "— W4A8 int8 kernels do not work on sm_75; keep H3/LTX on the 3070 (there is no Colab video fallback group). Full error above / in colab/comfyui.log.")

## Optional probe: PC ⇄ worker reachability over Tailscale

Run only if you added the `TAILSCALE_AUTHKEY` secret. Re-runs the bootstrap's mesh step, then prints exactly what to dial from the PC:
- `curl http://<ip printed>:8120/ping` → direct netstack inbound works → `public_base_url` can be the raw IP.
- otherwise the printed `https://…ts.net` URL (`tailscale serve`) is the path — that is what `bootstrap.sh` registers.

In [ ]:
# @title 5 · (optional) Tailscale inbound probe { display-mode: "form" }
import subprocess
if not __import__("os").environ.get("TAILSCALE_AUTHKEY"):
    print("TAILSCALE_AUTHKEY not set — skipped")
else:
    subprocess.run(["bash", "-c",
        "cd /content/kinoloom && source colab/colab_env.sh; "
        "tailscale ip -4; tailscale status --json | python3 -c "
        \"import json,sys;print('serve host:', json.load(sys.stdin)['Self']['DNSName'].rstrip('.'))\"; "
        "tailscale serve status 2>/dev/null | head -5"], check=False)
    print("\nfrom the PC run:  curl http://<ip-above>:8120/ping   (or the https serve URL)")